In [1]:
# from IPython.display import HTML
# from brax.v1.io import html

import torch 

import jax
import jax.numpy as jnp

# from qdax import environments

import brax
from tqdm import tqdm

import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
from brax.io import torch as io_torch
import random

class TorchWrapper:
    def __init__(self, env, num_envs):
        
        self.env = env
        self.num_envs = num_envs
        self.state_dim = env.observation_size
        self.action_dim = env.action_size
        
        
        self.reset_fn = jax.jit(jax.vmap(env.reset))
        self.step_fn = jax.jit(jax.vmap(env.step))
        
        # self.reset_fn = jax.vmap(env.reset)
        # self.step_fn = jax.vmap(env.step)
        
    
    def reset(self, seed=None):
        
        random_key = jax.random.PRNGKey(random.randint(0, 99999999))
        keys = jax.random.split(random_key, num=self.num_envs)
        # random_key, subkey = jax.random.split(random_key)
        if seed == 1:
            keys = jax.random.PRNGKey(random.randint(0, 99999999))
        state = self.reset_fn(keys)
        self.state = state
        return io_torch.jax_to_torch(state.obs)
    
    def step(self, action: torch.Tensor):
        
        action = io_torch.torch_to_jax(action)
        next_state = self.step_fn(self.state, action)
        observation, reward, done, state_descriptor = next_state.obs, next_state.reward, next_state.done, next_state.info['state_descriptor']
        observation = io_torch.jax_to_torch(observation)
        reward = io_torch.jax_to_torch(reward)
        done = io_torch.jax_to_torch(done)
        # print('jax:', state_descriptor)
        state_descriptor = io_torch.jax_to_torch(state_descriptor)
        # print('torch:', state_descriptor)
        
        self.state = next_state
        
        return observation, reward, done, {'state_descriptor': state_descriptor}
        

---

In [3]:
from brax import base
from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from etils import epath
import jax
from jax import numpy as jp


class PointMaze(PipelineEnv):

    def __init__(self):

        backend='mjx' #'generalized'
        kwargs={}
        
        path = 'mazes/point_mass_maze_empty.xml'
        sys = mjcf.load(path)
        
        n_frames = 1
        kwargs['n_frames'] = kwargs.get('n_frames', n_frames)
        super().__init__(sys=sys, backend=backend, **kwargs)
        
        # path = epath.resource_path('brax') / 'envs/assets/half_cheetah.xml'
        
        
    def reset(self, rng: jax.Array) -> State:
        """Resets the environment to an initial state."""
        rng, rng1, rng2 = jax.random.split(rng, 3)

        _reset_noise_scale = 0.001

        low, hi = -_reset_noise_scale, _reset_noise_scale
        qpos = self.sys.init_q + jax.random.uniform(
            rng1, (self.sys.q_size(),), minval=low, maxval=hi
        )
        qvel = hi * jax.random.normal(rng2, (self.sys.qd_size(),))
        
        pipeline_state = self.pipeline_init(qpos, qvel)

        
        obs = self._get_obs(pipeline_state)
        reward, done, zero = jp.zeros(3)
        metrics = {
            'x_position': zero,
            'y_position': zero,
        }
        return State(pipeline_state, obs, reward, done, metrics)
    
    def step(self, state: State, action: jax.Array) -> State:
        pipeline_state0 = state.pipeline_state
        assert pipeline_state0 is not None
        pipeline_state = self.pipeline_step(pipeline_state0, action)

        # x_velocity = (
        #     pipeline_state.x.pos[0, 0] - pipeline_state0.x.pos[0, 0]
        # ) / self.dt
        # forward_reward = self._forward_reward_weight * x_velocity
        # ctrl_cost = self._ctrl_cost_weight * jp.sum(jp.square(action))

        obs = self._get_obs(pipeline_state)
        # reward = forward_reward - ctrl_cost
        reward = 0
        # state.metrics.update(
        #     x_position=pipeline_state.x.pos[0, 0],
        #     x_velocity=x_velocity,
        #     reward_run=forward_reward,
        #     reward_ctrl=-ctrl_cost,
        # )

        return state.replace(pipeline_state=pipeline_state, obs=obs, reward=reward)
    
    def _get_obs(self, pipeline_state):
        return jnp.concatenate((pipeline_state.qpos, pipeline_state.qvel), axis=-1)

In [32]:
env = PointMaze()

# reset_fn = jax.jit(jax.vmap(env.reset))
# step_fn = jax.jit(jax.vmap(env.step))

reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

In [33]:
random_key = jax.random.PRNGKey(random.randint(0, 99999999))
# keys = jax.random.split(random_key, num=1)
state = reset_fn(random_key)

In [34]:
rollout = []

for _ in tqdm(range(10)):
    # action = jnp.array([[0.2, 0.2], [0.2, 0.2]])
    # action = jnp.full((1, 2), 1.)
    action = jnp.array([0.5, 1.0])
    next_state = step_fn(state, action)

    state = next_state
    
    s = state.pipeline_state
    # s.x.pos = s.x.pos[..., :2]
    # s.x.rot = s.x.rot[..., :2]
    
    rollout.append(s)

100%|██████████| 10/10 [00:08<00:00,  1.16it/s]


In [7]:
brax.v1.envs.env.State(
    qp=brax.v1.physics.base.QP(
        pos=state.pipeline_state.qpos,
        # rot=eval_env.state.qp.rot.reshape(8, 4),
        vel=state.pipeline_state.qvel,
        # ang=eval_env.state.qp.ang,
    ),
    obs=state.obs,
    reward=state.reward,
    done=state.done,
    info=state.info,
)

AttributeError: module 'brax' has no attribute 'v1'

In [45]:
from brax.io import html
from IPython.display import HTML

print(html.render(env.sys, rollout, colab=True))

<!DOCTYPE html>
<html>

  <head>
    <title>Brax visualizer</title>
    <link rel="shortcut icon" type="image/x-icon" href="/favicon.ico">
  </head>

  <style>
    body {
      margin: 0;
      padding: 0;
    }

    #brax-viewer {
      height: 480px;
      margin: 0;
      padding: 0;
    }
  </style>
  <script async src="https://unpkg.com/es-module-shims@1.6.3/dist/es-module-shims.js"></script>

  <script type="importmap">
    {
      "imports": {
        "three": "https://unpkg.com/three@0.150.1/build/three.module.js",
        "three/addons/": "https://unpkg.com/three@0.150.1/examples/jsm/",
        "lilgui": "https://cdn.jsdelivr.net/npm/lil-gui@0.18.0/+esm",
        "viewer": "https://cdn.jsdelivr.net/gh/google/brax@v0.12.1/brax/visualizer/js/viewer.js"
      }
    }
  </script>

  <script src="https://unpkg.com/pako@2.1.0/dist/pako.min.js"></script>

  <script type="application/javascript">
  var system = "eJzNVduO2kAM/RWUZ0Aej2/Tn2il7dsKVVSbbqMFgkikbov493oSoGwXVkUlVfMwUuwZ+/j4j

In [39]:
html.render(env.sys, rollout, colab=True)

'<!DOCTYPE html>\n<html>\n\n  <head>\n    <title>Brax visualizer</title>\n    <link rel="shortcut icon" type="image/x-icon" href="/favicon.ico">\n  </head>\n\n  <style>\n    body {\n      margin: 0;\n      padding: 0;\n    }\n\n    #brax-viewer {\n      height: 480px;\n      margin: 0;\n      padding: 0;\n    }\n  </style>\n  <script async src="https://unpkg.com/es-module-shims@1.6.3/dist/es-module-shims.js"></script>\n\n  <script type="importmap">\n    {\n      "imports": {\n        "three": "https://unpkg.com/three@0.150.1/build/three.module.js",\n        "three/addons/": "https://unpkg.com/three@0.150.1/examples/jsm/",\n        "lilgui": "https://cdn.jsdelivr.net/npm/lil-gui@0.18.0/+esm",\n        "viewer": "https://cdn.jsdelivr.net/gh/google/brax@v0.12.1/brax/visualizer/js/viewer.js"\n      }\n    }\n  </script>\n\n  <script src="https://unpkg.com/pako@2.1.0/dist/pako.min.js"></script>\n\n  <script type="application/javascript">\n  var system = "eJzNVduO2kAM/RWUZ0Aej2/Tn2il7dsKVVSb

In [40]:
rollout

[State(ne=0, nf=0, nl=2, nefc=22, ncon=5, solver_niter=Array(0, dtype=int32), time=Array(0.02, dtype=float32), qpos=Array([ 0.00065741, -0.00058353], dtype=float32), qvel=Array([0.00402666, 0.00611475], dtype=float32), act=Array([], shape=(0,), dtype=float32), qacc_warmstart=Array([0.16346075, 0.33381423], dtype=float32), ctrl=Array([0.5, 1. ], dtype=float32), qfrc_applied=Array([0., 0.], dtype=float32), xfrc_applied=Array([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.]], dtype=float32), eq_active=Array([], shape=(0,), dtype=uint8), mocap_pos=Array([], shape=(0, 3), dtype=float32), mocap_quat=Array([], shape=(0, 4), dtype=float32), qacc=Array([0.16346075, 0.33381423], dtype=float32), act_dot=Array([], shape=(0,), dtype=float32), userdata=Array([], shape=(0,), dtype=float32), sensordata=Array([], shape=(0,), dtype=float32), xpos=Array([[ 0.        ,  0.        ,  0.        ],
        [ 0.00057688, -0.00070583,  0.01      ]], dtype=float32), xquat=Array([[1., 0., 0., 0.],
   